## PyINE-v1 code execution trace dataset repackaging for HuggingFace

This notebook loads a local trace dataset (stored in LMDB shards) and pushes it to a HuggingFace
dataset repository in Parquet format. Each row corresponds to a single code execution trace, with
structured columns for easy filtering and JSON-serialized columns for the full execution trace data.

**Prerequisites:** `pip install datasets huggingface_hub` and `huggingface-cli login`.

In [ ]:
import collections.abc
import json

import datasets
import huggingface_hub
import tqdm

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.data.utils.splits
import pyine.utils.code.execution
import pyine.utils.filesystem
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()

In [ ]:
# ------------ SETTINGS ------------
SOURCE_DATASET_NAME = "TACO"
SOURCE_DATASET_REPO = "BAAI/TACO"
TRACE_DATASET_PATTERN = "v1.5/10s10t.*of000026.*.lmdb"
EXPECTED_PART_COUNT = 26

HF_REPO_NAME = "your-username/pyine-traces-taco"  # <-- change this
MAX_SHARD_SIZE = "1GB"
INCLUDE_TRACED_STEPS = True
# ----------------------------------

### Load trace dataset shards

In [ ]:
dataset_paths = sorted(
    pyine.data.traces.dataset_utils.get_matching_dataset_paths(
        source_dataset_name=SOURCE_DATASET_NAME,
        pattern=TRACE_DATASET_PATTERN,
    )
)
assert len(dataset_paths) == EXPECTED_PART_COUNT, f"expected {EXPECTED_PART_COUNT} shards, found {len(dataset_paths)}"

readers: list[pyine.data.traces.dataset_reader.DatasetReader] = []
for path in tqdm.tqdm(dataset_paths, desc="loading shards"):
    readers.append(pyine.data.traces.dataset_reader.DatasetReader(path))

total_traces = sum(len(r) for r in readers)
print(f"loaded {len(readers)} shards with {total_traces:,} total traces")

In [ ]:
# load the pre-computed dataset split to map problems to pyine subsets (train/valid/test)
split_result = pyine.data.utils.splits.get_dataset_split_result(SOURCE_DATASET_NAME)
print(f"split subsets: {split_result.config.subset_names}")
print(f"split covers {len(split_result.subset_assignments):,} problems")

# build a quick summary of subset sizes
subset_counts = {}
for subset_name in split_result.subset_assignments.values():
    subset_counts[subset_name] = subset_counts.get(subset_name, 0) + 1
for subset_name, count in sorted(subset_counts.items()):
    print(f"  {subset_name}: {count:,} problems")

### Define trace-to-row conversion

Each trace is flattened into a dict with:
- **Structured columns** for identity, code, I/O, tags, and complexity metrics (easy to filter/browse on HF)
- **JSON string columns** for deeply nested data (execution trace steps, code blocks, test pairs)

In [ ]:
def _safe_json_dumps(value: object) -> str:
    """Serializes a value to a JSON string, falling back to repr for non-serializable types."""
    if value is None:
        return "null"
    try:
        return json.dumps(value, ensure_ascii=False)
    except (TypeError, ValueError):
        return json.dumps(repr(value), ensure_ascii=False)


def trace_to_hf_row(
    trace: pyine.utils.code.execution.TraceResult,
    problem: pyine.data.traces.dataset_utils.CodingProblem,
    pyine_subset: str | None,
    include_traced_steps: bool = True,
) -> dict:
    """Converts a TraceResult + CodingProblem into a flat dict suitable for HF datasets."""
    trace_id = pyine.data.traces.dataset_utils.TraceIdentifier.from_string(trace.identifier)
    row = {
        # ---- trace identity ----
        "identifier": trace.identifier,
        "dataset_name": trace_id.dataset,
        "orig_subset": trace_id.subset,
        "pyine_subset": pyine_subset,
        "problem_idx": trace_id.problem_idx,
        "solution_idx": trace_id.solution_idx,
        "test_idx": trace_id.test_idx,
        "augment_category": trace_id.augment_category,
        "augment_idx": trace_id.augment_idx,
        "is_augmented": trace_id.is_augmented,
        # ---- code + execution results ----
        "code_string": trace.code_string,
        "entrypoint_name": trace.entrypoint_name,
        "inputs_json": _safe_json_dumps(trace.inputs),
        "expected_output_json": _safe_json_dumps(trace.expected_output),
        "return_value_json": _safe_json_dumps(trace.return_value),
        "stdout": trace.stdout,
        "stderr": trace.stderr,
        "tags": trace.tags,
        # ---- exception info ----
        "exception_type": trace.exception.type if trace.exception else None,
        "exception_message": trace.exception.message if trace.exception else None,
        # ---- step counts ----
        "valid_step_count": trace.valid_step_count,
        "total_step_count": trace.total_step_count,
        # ---- complexity metrics (flattened as individual numeric columns) ----
        **{f"complexity_{key}": val for key, val in trace.complexity_metrics.as_dict().items()},
        # ---- problem-level data ----
        "problem_statement": problem.problem_statement,
        "problem_tags": problem.problem_tags,
        "problem_entrypoint_name": problem.entrypoint_name,
        "problem_is_banned": problem.is_banned,
        "test_inout_pairs_json": _safe_json_dumps(
            [list(pair) for pair in problem.test_inout_pairs],
        ),
        # ---- heavy data (JSON-serialized) ----
        "trace_metadata_json": _safe_json_dumps(trace.metadata),
        "code_blocks_json": json.dumps(
            {
                key: {
                    "type": cb.type.value,
                    "name": cb.name,
                    "depth": cb.depth,
                    "parent_line": cb.parent_line,
                    "start_line": cb.start_line,
                    "end_line": cb.end_line,
                }
                for key, cb in trace.code_blocks.items()
            },
            ensure_ascii=False,
        ),
    }
    if include_traced_steps:
        # serialize the full execution trace; this is the largest field per row
        trace_dump = trace.model_dump(mode="json")
        row["traced_steps_json"] = json.dumps(trace_dump["traced_steps"], ensure_ascii=False)
        row["traced_steps_map_json"] = json.dumps(trace_dump["traced_steps_map"], ensure_ascii=False)
    return row

### Build HF dataset via generator

The generator iterates over all shards and traces, converting each to a flat dict.
`Dataset.from_generator` handles Arrow/Parquet conversion and schema inference.

In [ ]:
subset_assignments = split_result.subset_assignments


def trace_dataset_generator() -> collections.abc.Iterator[dict]:
    """Yields one HF-compatible dict per trace across all loaded shards."""
    progress = tqdm.tqdm(total=total_traces, desc="repackaging traces")
    for reader in readers:
        problem_cache: dict[str, pyine.data.traces.dataset_utils.CodingProblem] = {}
        for trace_idx in range(len(reader)):
            trace = reader[trace_idx]
            problem_key = reader.trace_key_to_problem_key[trace.identifier]
            if problem_key not in problem_cache:
                problem_cache[problem_key] = reader.get_problem_data(trace_idx)
            pyine_subset = subset_assignments.get(problem_key)
            yield trace_to_hf_row(
                trace=trace,
                problem=problem_cache[problem_key],
                pyine_subset=pyine_subset,
                include_traced_steps=INCLUDE_TRACED_STEPS,
            )
            progress.update(1)
    progress.close()

In [ ]:
hf_cache_dir = pyine.utils.filesystem.get_data_cache_subdir("hf_trace_rpkg")
hf_dataset = datasets.Dataset.from_generator(
    trace_dataset_generator,
    cache_dir=str(hf_cache_dir),
)
print(f"created HF dataset with {len(hf_dataset):,} rows and {len(hf_dataset.column_names)} columns")
print(f"columns: {hf_dataset.column_names}")

### Preview a sample row

In [ ]:
sample = hf_dataset[0]
json_cols = [col for col in sample if col.endswith("_json")]
for key, value in sample.items():
    if key in json_cols:
        preview = str(value)[:120] + "..." if len(str(value)) > 120 else str(value)
        print(f"  {key}: {preview}")
    elif isinstance(value, str) and len(value) > 200:
        print(f"  {key}: {value[:200]}...")
    else:
        print(f"  {key}: {value}")

### Build dataset card and push to Hugging Face Hub

In [ ]:
# collect dataset-level metadata from the first shard to attach to the HF dataset info
shard_metadata = readers[0].metadata
metadata_fields_too_big = {"key_map", "installed_packages"}
dataset_level_metadata = {key: val for key, val in shard_metadata.items() if key not in metadata_fields_too_big}
dataset_level_metadata["source_dataset_repo"] = SOURCE_DATASET_REPO
dataset_level_metadata["total_shards"] = len(readers)
dataset_level_metadata["total_traces"] = total_traces
dataset_level_metadata["include_traced_steps"] = INCLUDE_TRACED_STEPS
dataset_level_metadata["pyine_subset_names"] = split_result.config.subset_names
dataset_level_metadata["pyine_split_seed"] = split_result.config.seed
dataset_level_metadata["pyine_split_probs"] = {
    name: float(prob) for name, prob in split_result.config.subset_assign_prob_map.items()
}

# attach metadata to the dataset info so it's stored alongside the parquet files
hf_dataset.info.description = (
    f"PyINE-v1 execution trace dataset derived from {SOURCE_DATASET_NAME} "
    f"(https://huggingface.co/datasets/{SOURCE_DATASET_REPO}). "
    f"Contains {total_traces:,} traces across {len(readers)} shards."
)
hf_dataset.info.dataset_name = HF_REPO_NAME.split("/")[-1]

# store custom metadata fields (shown in the dataset card on HF)
if hf_dataset.info.config_kwargs is None:
    hf_dataset.info.config_kwargs = {}
hf_dataset.info.config_kwargs["metadata"] = {
    key: str(val) if not isinstance(val, (str, int, float, bool)) else val
    for key, val in dataset_level_metadata.items()
}
# source_datasets is recognized by HF and rendered as a link on the dataset page
hf_dataset.info.config_kwargs["source_datasets"] = [SOURCE_DATASET_REPO]

print(f"dataset-level metadata keys: {list(dataset_level_metadata.keys())}")

In [ ]:
# build split statistics for the card
subset_trace_counts: dict[str, int] = {}
for reader in readers:
    for meta in reader.trace_metadata:
        problem_key = reader.trace_key_to_problem_key[meta.identifier]
        pyine_sub = subset_assignments.get(problem_key, "unassigned")
        subset_trace_counts[pyine_sub] = subset_trace_counts.get(pyine_sub, 0) + 1

split_table_rows = "\n".join(
    f"| {name} | {subset_counts.get(name, 0):,} | {subset_trace_counts.get(name, 0):,} |"
    for name in split_result.config.subset_names
)


def _get_hf_size_category(count: int) -> str:
    """Returns the HuggingFace size category string for a given item count."""
    if count >= 1_000_000_000_000:
        return "n>1T"
    if count >= 1_000_000_000:
        return "1B<n<1T"
    if count >= 100_000_000:
        return "100M<n<1B"
    if count >= 10_000_000:
        return "10M<n<100M"
    if count >= 1_000_000:
        return "1M<n<10M"
    if count >= 100_000:
        return "100K<n<1M"
    if count >= 10_000:
        return "10K<n<100K"
    if count >= 1_000:
        return "1K<n<10K"
    return "n<1K"


card_data = huggingface_hub.DatasetCardData(
    language="en",
    license="cc-by-4.0",
    source_datasets=[SOURCE_DATASET_REPO],
    tags=["code", "code-execution", "execution-traces", "python", "code-analysis"],
    task_categories=["text-generation", "code-execution"],
    size_categories=[_get_hf_size_category(total_traces)],
    pretty_name=f"PyINE-v1 Execution Traces ({SOURCE_DATASET_NAME})",
)

traced_steps_note = (
    "The full execution trace (variable snapshots at each step) is included in "
    "`traced_steps_json` and `traced_steps_map_json`."
    if INCLUDE_TRACED_STEPS
    else "Full execution trace data (`traced_steps_json`) was **excluded** from this export "
    "to reduce size. Only step counts are available."
)

card_content = f"""\
# PyINE-v1 Execution Traces ({SOURCE_DATASET_NAME})

This dataset contains **{total_traces:,}** Python code execution traces generated by the
[PyINE](https://github.com/TODO/pyine) framework from solutions in the
[{SOURCE_DATASET_NAME}]({f"https://huggingface.co/datasets/{SOURCE_DATASET_REPO}"}) dataset.

Each row is a single execution trace: one code solution executed against one test input,
capturing the full sequence of variable states at every line of execution.

## Dataset structure

### Splits

Traces are assigned to PyINE splits at the **problem level** (all traces for a given problem
share the same split), using stratified random assignment.

| Split | Problems | Traces |
|-------|----------|--------|
{split_table_rows}

### Columns

**Identity:**
| Column | Type | Description |
|--------|------|-------------|
| `identifier` | `string` | Full trace identifier (e.g. `TACO/train/p000001/s0000/t0001`) |
| `dataset_name` | `string` | Source dataset name |
| `orig_subset` | `string` | Subset from the original (parent) dataset |
| `pyine_subset` | `string` | Subset assigned by PyINE's stratified split |
| `problem_idx` | `int` | Problem index in the source dataset |
| `solution_idx` | `int` | Solution index for this problem |
| `test_idx` | `int` | Test case index |
| `augment_category` | `string?` | Code augmentation type(s) applied, if any |
| `augment_idx` | `int?` | Augmentation instance index, if any |
| `is_augmented` | `bool` | Whether this trace uses augmented code |

**Code and execution results:**
| Column | Type | Description |
|--------|------|-------------|
| `code_string` | `string` | Python source code that was executed |
| `entrypoint_name` | `string?` | Name of the called entry function, if any |
| `inputs_json` | `string` | JSON-serialized input arguments |
| `expected_output_json` | `string` | JSON-serialized expected output |
| `return_value_json` | `string` | JSON-serialized actual return value |
| `stdout` | `string` | Captured standard output |
| `stderr` | `string` | Captured standard error |
| `tags` | `list[string]` | Tags (source, difficulty, execution outcome, augmentation) |
| `exception_type` | `string?` | Exception type name if the execution raised one |
| `exception_message` | `string?` | Exception message text, if any |
| `valid_step_count` | `int` | Number of in-scope execution steps |
| `total_step_count` | `int` | Total recorded steps (including out-of-scope) |

**Complexity metrics** (from [radon](https://radon.readthedocs.io/)):
`complexity_cyclomatic_complexity_avg`, `complexity_cyclomatic_complexity_max`,
`complexity_cyclomatic_complexity_sum`, `complexity_loc`, `complexity_lloc`,
`complexity_sloc`, `complexity_comments`, `complexity_multi`, `complexity_blank`,
`complexity_halstead_volume`, `complexity_halstead_difficulty`,
`complexity_halstead_effort`, `complexity_maintainability_index`

**Problem-level data:**
| Column | Type | Description |
|--------|------|-------------|
| `problem_statement` | `string` | Full problem description from the source dataset |
| `problem_tags` | `list[string]` | Problem tags (difficulty, source, etc.) |
| `problem_entrypoint_name` | `string?` | Expected function name to implement, if specified |
| `problem_is_banned` | `bool` | Whether this problem was flagged for data issues |
| `test_inout_pairs_json` | `string` | JSON-serialized list of all test input/output pairs |

**Serialized data (JSON strings):**
| Column | Type | Description |
|--------|------|-------------|
| `trace_metadata_json` | `string` | Execution environment metadata |
| `code_blocks_json` | `string` | Code block structure (if/for/while/function boundaries) |
| `traced_steps_json` | `string` | Full execution trace: variable snapshots at each step |
| `traced_steps_map_json` | `string` | Map from code location to step indices |

{traced_steps_note}

## Usage

```python
import datasets

ds = datasets.load_dataset("{HF_REPO_NAME}")

# filter by split
train_ds = ds.filter(lambda row: row["pyine_subset"] == "train")

# access a trace
trace = ds[0]
print(trace["code_string"])
print(f"steps: {{trace['valid_step_count']}}")
```

## Source

Derived from [{SOURCE_DATASET_NAME}](https://huggingface.co/datasets/{SOURCE_DATASET_REPO})
using the [PyINE](https://github.com/TODO/pyine) execution tracing framework.
"""

dataset_card = huggingface_hub.DatasetCard(card_content)
dataset_card.data = card_data
print(str(dataset_card)[:3000] + "\n...")

In [ ]:
hf_dataset.push_to_hub(
    repo_id=HF_REPO_NAME,
    max_shard_size=MAX_SHARD_SIZE,
)
dataset_card.push_to_hub(HF_REPO_NAME, repo_type="dataset")
print(f"pushed to https://huggingface.co/datasets/{HF_REPO_NAME}")